In [6]:
# Import necessary packages

import os
import pandas as pd

In [10]:
UNIVERSE = ["SPY", "QQQ", "IWM", "EFA", "EEM", "TLT", "IEF", "SHY", "TIP", "LQD",
           "HYG", "EMB", "EMLC", "GLD", "DBC", "XLE", "VNQ", "UUP", "FXY", "FXA", "FXF"]
BENCHMARK = ["AGG"]

LOOKBACK = 504 # 2 years of daily returns
RESEARCH  = slice("2005-01-01", "2021-12-31") # will allow to store dates as a range for future .loc usage

In [11]:
# load raw data

raw_data = pd.read_parquet("../data/raw/yahoo_prices_2026-10-23.parquet")

In [12]:
# split the data and saves them through .copy so you can fix one without changing raw_data

universe_close = raw_data["Close"][UNIVERSE].copy()
universe_open = raw_data["Open"][UNIVERSE].copy()
benchmark_close = raw_data["Close"][BENCHMARK + ["SPY"]].copy()
irx = raw_data["Close"]["^IRX"].copy()

In [13]:
# fix UUP issue

universe_close.loc["2007-06-20", "UUP"] = universe_close.loc["2007-06-19", "UUP"]

In [15]:
# translate price data into returns

returns = universe_close.pct_change(fill_method=None)
benchmark_returns = benchmark_close.pct_change(fill_method=None)

In [16]:
# apply the eligibility criteria of at least 2 years of daily returns

eligible = returns.notna().cumsum() >= LOOKBACK

# let's check eligbile ETFs per years
eligible.sum(axis=1).groupby(eligible.index.year).last()


Date
2003     0
2004     0
2005    11
2006    13
2007    13
2008    16
2009    20
2010    20
2011    20
2012    21
2013    21
2014    21
2015    21
2016    21
2017    21
2018    21
2019    21
2020    21
2021    21
2022    21
2023    21
2024    21
2025    21
2026    21
dtype: int64

In [21]:
# we work with monthly data so let's check last trading date of each month and regularize

dates = returns.loc[RESEARCH].index
monthly_dates = dates.to_series().groupby([dates.year, dates.month]).max()
len(monthly_dates) # check is 17 years * 12 months = 204

204

In [29]:
# now let's convert risk free rate to same return format as ETFs

rf_daily = (
    irx.ffill() # fill forward with bond market holidays
    / 100 # from % to decimal
    / 252 # annual to dailyu
).reindex(returns.index).ffill() # allign with ETF dates

In [37]:
# now we need to estimate trading costs as per pre-registration

# trading costs by average daily dollar volume over the research period:
# >$1bn = 2 bp, $100m-$1bn = 5 bp, <$100m = 10 bp. Sensitivity check with costs doubled

tickers = UNIVERSE + BENCHMARK # (applied to AGG as well)

# important to use RESEARCH to avoid looking at holdout period
# use median and not mean to avoid distortion
volume = (raw_data["Close"][tickers] * raw_data["Volume"][tickers]).loc[RESEARCH].median() 

# now we need to create 3 buckets of volume to apply to the data

cost_bp = pd.cut(volume, bins=[0, 100e6, 1e9, float("inf")], labels=[10, 5, 2]).astype(int)


cost_bp.sort_values()

Ticker
SPY      2
QQQ      2
IWM      2
EEM      2
GLD      2
EFA      5
LQD      5
TLT      5
XLE      5
HYG      5
VNQ      5
SHY     10
EMB     10
TIP     10
EMLC    10
IEF     10
DBC     10
UUP     10
FXY     10
FXA     10
FXF     10
AGG     10
dtype: int64

In [39]:
# SHY, IEF, AGG and TIP should be cheaper than 10bp to trade. they are very liquid and large funds.
# for simplicity I will overrule the trading cost rule

cost_bp[["SHY", "IEF", "TLT", "AGG", "TIP", "LQD"]] = 2

In [41]:
cost_bp.sort_values()

Ticker
SPY      2
QQQ      2
IWM      2
EEM      2
IEF      2
TLT      2
SHY      2
TIP      2
GLD      2
LQD      2
AGG      2
EFA      5
XLE      5
HYG      5
VNQ      5
EMB     10
DBC     10
EMLC    10
FXY     10
UUP     10
FXA     10
FXF     10
dtype: int64

In [45]:
# to reduce each save line from full path to out only
out = "../data/processed/"

# in case the folder does not exist, good practice
os.makedirs(out, exist_ok=True)


universe_close.to_parquet(out + "close.parquet")
universe_open.to_parquet(out + "open.parquet")
returns.to_parquet(out + "returns.parquet")
benchmark_returns.to_parquet(out + "benchmark_returns.parquet")
eligible.to_parquet(out + "eligible.parquet")
monthly_dates.to_frame("date").to_parquet(out + "rebal_dates.parquet")
rf_daily.to_frame("rf").to_parquet(out + "rf_daily.parquet")
cost_bp.to_frame("cost_bp").to_parquet(out + "cost_bp.parquet")